In [1]:
import json
from glob import glob

In [2]:
path = "../results/"
results = glob(path + "*/*/result.json")

In [3]:
successful_runs = []
for result in results:
    try:
        with open(result, "r") as f:
            data = json.load(f)
            # print(data)
            if data and 'result' in data and data['result'] and  'success' in data['result'] and data['result']['success']:
                data['path'] = result.replace("/result.json", "")
                successful_runs.append(data)
    except Exception as e:
        print(result)
        print(data)
        raise e

In [4]:
len(successful_runs)

4101

In [5]:
tools=[
            {
                "type": "function",
                "function": {
                    "name": "step_browser",
                    "description": "Execute an action in the web browser environment and get the resulting observation.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "action": {
                                "type": "object",
                                "description": """Browser action to execute. Available actions:

1. CLICK ELEMENT - Click on any clickable element
   {"action": "click", "target": "semantic_id"}
   - target: The semantic ID of the element to click (REQUIRED)

2. TYPE TEXT - Type text into input fields with optional Enter key press
   {"action": "type", "target": "semantic_id", "text": "content", "enter": true}
   - target: The semantic ID of the input element (REQUIRED)
   - text: The text content to type (REQUIRED)
   - enter: Whether to press Enter after typing (OPTIONAL, default: false)

3. HOVER ELEMENT - Hover over an element to trigger tooltips or dropdowns
   {"action": "hover", "target": "semantic_id"}
   - target: The semantic ID of the element to hover over (REQUIRED)

4. SELECT OPTION - Select an option from dropdown/select elements
   {"action": "select", "target": "semantic_id", "value": "option_value"}
   - target: The semantic ID of the select element (REQUIRED)
   - value: The value of the option to select (REQUIRED)

5. CLEAR INPUT - Clear the content of an input element
   {"action": "clear", "target": "semantic_id"}
   - target: The semantic ID of the input element to clear (REQUIRED)

6. KEY PRESS - Press keyboard keys, optionally on specific elements
   {"action": "key_press", "key": "Enter", "target": "semantic_id"}
   - key: The key to press (e.g., "Enter", "Escape", "Tab", "ArrowDown") (REQUIRED)
   - target: The semantic ID of element to focus before key press (OPTIONAL)

7. NAVIGATE TO URL - Navigate to a specific URL in current tab
   {"action": "goto_url", "url": "https://example.com"}
   - url: The URL to navigate to (REQUIRED)

8. GO BACK - Navigate back in browser history
   {"action": "back"}
   - No parameters required

9. GO FORWARD - Navigate forward in browser history
   {"action": "forward"}
   - No parameters required

10. REFRESH PAGE - Reload the current page
    {"action": "refresh"}
    - No parameters required

11. NEW TAB - Open a new browser tab
    {"action": "new_tab", "url": "https://example.com"}
    - url: URL to open in the new tab (OPTIONAL, opens blank tab if not provided)

12. SWITCH TAB - Switch to a different browser tab
    {"action": "switch_tab", "tab_id": 0}
    - tab_id: The ID number of the tab to switch to (REQUIRED, starts from 0)

13. CLOSE TAB - Close a specific browser tab
    {"action": "close_tab", "tab_id": 0}
    - tab_id: The ID number of the tab to close (REQUIRED, starts from 0)

14. TERMINATE TASK - End the task with final answer
    {"action": "terminate", "answer": "final answer or result"}
    - answer: Your final answer or result for the task (OPTIONAL, use empty string if no specific answer)""",
                                "properties": {
                                    "action": {"type": "string", "description": "Type of action to perform", "enum": ["click", "type", "hover", "select", "clear", "key_press", "goto_url", "back", "forward", "refresh", "new_tab", "switch_tab", "close_tab", "terminate"]},
                                    "target": {"type": "string", "description": "Semantic ID of the element to interact with (use data-semantic-id attribute)"},
                                    "text": {"type": "string", "description": "Text to type (for type action)"},
                                    "enter": {"type": "boolean", "description": "Whether to press Enter after typing (for type action)"},
                                    "value": {"type": "string", "description": "Value to select (for select action)"},
                                    "key": {"type": "string", "description": "Key to press (for key_press action) - e.g., 'Enter', 'Escape', 'Tab', 'ArrowDown'"},
                                    "url": {"type": "string", "description": "URL to navigate to (for goto_url/new_tab actions)"},
                                    "tab_id": {"type": "integer", "description": "Tab ID for tab operations (starts from 0)"},
                                    "answer": {"type": "string", "description": "Final answer (for terminate action)"},
                                },
                                "required": ["action"],
                            },
                        },
                        "required": ["action"],
                    },
                },
            },
        ]

In [6]:
import re
def transform_bedrock_to_openai(trace, system_prompt):
    # Create the new OpenAI format messages list
    openai_messages = [
        {"role": "system", "content": system_prompt}
    ]

    # Process conversation history
    prev_function_call = None
    for i, message in enumerate(trace["conversation_history"]):
        role = message["role"]

        # For user messages, treat them as tool responses of the previous assistant action
        if role == "user":
            if prev_function_call and i > 0:
                # This is a response to a tool call
                tool_response = {"role": "tool", "name": "step_browser", "content": message["content"][0]["text"] if len(message["content"]) > 0 else ""}
                openai_messages.append(tool_response)
            else:
                # This is the first user message, so include it as a regular user message
                user_message = {"role": "user", "content": message["content"][0]["text"] if len(message["content"]) > 0 else ""}
                openai_messages.append(user_message)
            prev_function_call = False

        # For assistant messages, extract the action as a function call
        elif role == "assistant":
            content = message["content"][0]["text"] if len(message["content"]) > 0 else ""

            # Look for ACTION pattern in the content
            action_match = re.search(r"ACTION: (.*?)(?:\n|$)", content)
            thought_match = re.search(r"THOUGHT: (.*?)(?:\nACTION:|$)", content, re.DOTALL)

            reasoning = ""
            if thought_match:
                reasoning = thought_match.group(1).strip()

            if action_match:
                action_json_str = action_match.group(1)
                try:
                    action_json = json.loads(action_json_str)

                    # Format the arguments as {"action": <extracted action>, "reasoning": <extracted reasoning>}
                    arguments = {"action": action_json}

                    # Create assistant message with tool call
                    assistant_message = {
                        "role": "assistant",
                        "content": reasoning,  # Include reasoning as content
                        "tool_calls": [
                            {
                                "function": {
                                    "name": "step_browser",  # Tool name from tool_system.txt
                                    # huggingface format, arguments is a dict
                                    "arguments": arguments,
                                }
                            }
                        ],
                    }

                    openai_messages.append(assistant_message)
                    prev_function_call = True
                except json.JSONDecodeError:
                    # If action isn't valid JSON, include the message as is
                    openai_messages.append({"role": "assistant", "content": content})
                    prev_function_call = False
            else:
                # If no action is found, include the message as is
                openai_messages.append({"role": "assistant", "content": content})
                prev_function_call = False

    # Create the output structure
    openai_data = {"messages": openai_messages, "tools": tools}

    return openai_data


In [7]:

system_prompt_template = open("../rl_web_agent/prompts/tool_system.txt", "r").read()
processed_messages = []
for run in successful_runs:
    trace = json.load(open(run['path'] + "/session.json", "r"))
    openai = transform_bedrock_to_openai(trace, system_prompt_template.format(objective=run['task_config']['intent']))
    processed_messages.append(openai)

In [8]:
with open("training_fix_hf_format.jsonl", "w") as f:
    for openai in processed_messages:
        f.write(json.dumps(openai) + "\n")

In [31]:
!aws s3 cp training_fix_hf_format.jsonl s3://your-bucket/rl_web_agent/sft_data_curation/data_fix/training.jsonl

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


upload: ./training_fix_hf_format.jsonl to s3://your-bucket/rl_web_agent/sft_data_curation/data_fix/training.jsonl


In [10]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

/home/leo/rl_web_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [90]:
qwen_25_chat_template = r"""
{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n' }}
    {%- endif %}
{%- endif %}
{%- for message in messages %}
    {%- if (message.role == "user") or (message.role == "system" and not loop.first) %}
        {{- '<|im_start|>' + message.role + '\n' + message.content + '<|im_end|>' + '\n' }}
    {%- elif message.role == "assistant" %}
        {%- generation -%}
        {{- '<|im_start|>' + message.role }}
        {%- if message.content %}
            {{- '\n' + message.content }}
        {%- endif %}
        {%- for tool_call in message.tool_calls %}
            {%- if tool_call.function is defined %}
                {%- set tool_call = tool_call.function %}
            {%- endif %}
            {{- '\n<tool_call>\n{"name": "' }}
            {{- tool_call.name }}
            {{- '", "arguments": ' }}
            {{- tool_call.arguments | tojson }}
            {{- '}\n</tool_call>' }}
        {%- endfor %}
        {{- '<|im_end|>\n' }}
        {%- endgeneration -%}
    {%- elif message.role == "tool" %}
        {%- if (loop.index0 == 0) or (messages[loop.index0 - 1].role != "tool") %}
            {{- '<|im_start|>user' }}
        {%- endif %}
        {{- '\n<tool_response>\n' }}
        {{- message.content }}
        {{- '\n</tool_response>' }}
        {%- if loop.last or (messages[loop.index0 + 1].role != "tool") %}
            {{- '<|im_end|>\n' }}
        {%- endif %}
    {%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
    {{- '<|im_start|>assistant\n' }}
{%- endif %}
"""
for openai in processed_messages:
    tokenized = tokenizer.apply_chat_template(openai['messages'], tools=openai['tools'], return_dict=True, return_assistant_tokens_mask=True, chat_template=qwen_25_chat_template)
    print(len(tokenized['input_ids']))
    print(sum(tokenized['assistant_masks']))

72453
1311
100921
896
103438
1589
86010
785
14250
526
28085
347
47008
922
52579
803
39306
642
6967
171
103518
1529


KeyboardInterrupt: 

72453

In [13]:
import json
x = json.loads(open("training.jsonl", "r").read().split("\n")[0])

FileNotFoundError: [Errno 2] No such file or directory: 'training.jsonl'

In [14]:
x

NameError: name 'x' is not defined

In [ ]:
from transformers import AutoTokenizer
llama_8b_chat_template = r"""

"""

In [12]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

In [24]:
tt = tokenizer.apply_chat_template(
        [
            {"role": "user", "content": "What is the capital of the moon?"},
            {"role": "assistant", "content": "The capital of the moon is the moon.", "tool_calls": [{"function": {"name": "step_browser", "arguments": {"action": "click", "target": "semantic_id"}}}]},
            {"role": "tool", "content": "The capital of the moon is the moon.", "name": "step_browser"},
        ],
        tools=tools,
        # tokenize=False,
        chat_template=llama_8b_chat_template,
        return_dict=True,
        return_assistant_tokens_mask=True,
        return_tensors='pt'
    )

In [30]:
import torch
print(tokenizer.decode(tt['input_ids'].masked_fill(~tt['assistant_masks'].to(torch.bool), tokenizer.eos_token_id)[0], skip_special_tokens=True))

assistant

The capital of the moon is the moon.{"name": "step_browser", "parameters": {"action": "click", "target": "semantic_id"}}


In [27]:
tt['input_ids'].masked_fill(~tt['assistant_masks'].to(torch.bool), tokenizer.eos_token_id)

tensor([[128009, 128009, 128009,  ..., 128009, 128009, 128009]])